<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 실습 1. Zero-shot / One-shot / Few-shot 비교

### ① 라이브러리 설치

In [ ]:
# Hugging Face의 transformers 라이브러리를 설치합니다.
# -q 옵션은 설치 메시지를 간단하게 표시합니다.
!pip -q install transformers accelerate sentencepiece

### ② 모델 불러오기
- 한국어 instruction-following이 가능한 비교적 작은 모델을 사용합니다.
- Colab GPU 사용을 권장합니다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 사용할 모델 이름
# Colab에서 실행하기 쉽도록 비교적 작은 모델을 사용합니다.
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 문장을 토큰으로 변환하는 tokenizer를 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 사전학습된 언어모델을 불러옵니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("모델 로딩 완료!")

### ③ LLM에게 질문하는 함수

In [ ]:
def ask_llm(prompt):
    """
    prompt를 모델에 전달하고
    생성된 답변만 반환하는 함수
    """

    # 채팅 모델이 이해할 수 있는 message 형식으로 만듭니다.
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 모델의 chat template에 맞는 문자열로 변환합니다.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 문자열을 모델이 처리할 수 있는 token ID로 변환합니다.
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 모델이 답변을 생성합니다.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,

            # 생성할 최대 토큰을 지정합니다.
            max_new_tokens=100,

            # 실험 결과를 쉽게 비교하기 위해
            # 무작위 sampling을 사용하지 않습니다.
            do_sample=False
        )

    # 입력 부분을 제외하고 새로 생성된 답변만 가져옵니다.
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    # token ID를 다시 사람이 읽을 수 있는 문자열로 변환합니다.
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer.strip()

### 실험 A. Zero-shot
예제를 하나도 보여주지 않고 바로 문제를 해결하게 합니다.

In [ ]:
review = "배송은 조금 늦었지만 제품의 품질은 정말 만족스럽습니다."

zero_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

리뷰: {review}

감정:
"""

result = ask_llm(zero_shot_prompt)

print("=== Zero-shot ===")
print(result)

### 실험 B. One-shot
이번에는 모델에게 정답 예제 1개를 먼저 보여줍니다.

In [ ]:
one_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시]

리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(one_shot_prompt)

print("=== One-shot ===")
print(result)

### 실험 C. Few-shot
이번에는 서로 다른 유형의 예제 3개를 제공합니다.

In [ ]:
few_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시 1]
리뷰: 가격도 저렴하고 성능도 매우 좋습니다.
감정: 긍정

[예시 2]
리뷰: 특별히 좋지도 나쁘지도 않습니다.
감정: 중립

[예시 3]
리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(few_shot_prompt)

print("=== Few-shot ===")
print(result)

## 실습 2. 좋은 프롬프트 구조 만들기

### 실습용 문서 준비

In [ ]:
document = """
대규모 언어 모델(LLM)은 방대한 텍스트 데이터로 학습된
인공지능 모델이다. 사용자의 질문에 답하거나 문서를 요약하고,
글을 작성하거나 프로그램 코드를 생성할 수 있다.

최근에는 LLM이 외부 문서를 검색하는 RAG,
외부 도구를 사용하는 Tool Calling,
여러 작업을 스스로 수행하는 AI Agent와 결합되고 있다.

하지만 LLM은 사실과 다른 내용을 생성하는 환각(Hallucination),
프롬프트 인젝션, 개인정보 유출 등의 문제를 일으킬 수 있다.
따라서 실제 시스템에서는 모델의 출력뿐 아니라
데이터, 권한, 도구 사용에 대한 검증도 중요하다.
"""

### 실험 A. 모호한 프롬프트

In [ ]:
prompt_a = f"""
이 문서를 잘 요약해줘.

문서:
{document}
"""

result_a = ask_llm(prompt_a)

print("=== A. 모호한 프롬프트 ===")
print(result_a)

### 실험 B. Role 추가
이번에는 모델에게 어떤 관점에서 작업할 것인지 알려줍니다.

In [ ]:
prompt_b = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 교수다.

[Task]
다음 문서를 요약하라.

[Document]
{document}
"""

result_b = ask_llm(prompt_b)

print("=== B. Role 추가 ===")
print(result_b)

### 실험 C. Role + Context + Task
이제 누구를 위한 결과인지와 정확히 무엇을 해야 하는지를 추가합니다.

In [ ]:
prompt_c = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 교수다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Document]
{document}
"""

result_c = ask_llm(prompt_c)

print("=== C. Role + Context + Task ===")
print(result_c)

### 실험 D. Constraints 추가
이제 모델이 지켜야 할 조건을 명시합니다.

In [ ]:
prompt_d = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Document]
{document}
"""

result_d = ask_llm(prompt_d)

print("=== D. Constraints 추가 ===")
print(result_d)

### 실험 E. Example 추가
이번에는 원하는 출력의 모습을 예제로 보여줍니다.

In [ ]:
prompt_e = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Example]
개념: 자연어처리(Natural Language Processing)
설명: 컴퓨터가 인간의 언어를 처리하는 AI 분야이다.

[Document]
{document}
"""

result_e = ask_llm(prompt_e)

print("=== E. Example 추가 ===")
print(result_e)

### 실험 F. Output Format 추가
마지막으로 출력 형식을 명확하게 지정합니다.

In [ ]:
prompt_f = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Example]
개념: 자연어처리(Natural Language Processing)
설명: 컴퓨터가 인간의 언어를 처리하는 AI 분야이다.

[Output Format]
Markdown 표로 작성한다.

열은 다음 세 개만 사용한다.

| 개념 | 설명 | 중요 이유 |

[Document]
{document}
"""

result_f = ask_llm(prompt_f)

print("=== F. 6요소 완성 ===")
print(result_f)

### 6요소를 함수로 만들기

In [ ]:
def make_prompt(role, context, task,
                constraints, example,
                output_format, document):

    """
    좋은 프롬프트의 6요소를 받아
    하나의 프롬프트로 조립합니다.
    """

    prompt = f"""
[Role]
{role}

[Context]
{context}

[Task]
{task}

[Constraints]
{constraints}

[Examples]
{example}

[Output Format]
{output_format}

[Document]
{document}
"""

    return prompt

이제 각 요소를 따로 정의합니다.

In [ ]:
role = """
컴퓨터공학 전공 학생을 가르치는 조교다.
"""

context = """
컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.
"""

task = """
문서에서 핵심 개념을 추출하여 요약하라.
"""

constraints = """
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
"""

example = """
개념: 자연어처리(Natural Language Processing)
설명: 컴퓨터가 인간의 언어를 처리하는 AI 분야이다.
"""

output_format = """
Markdown 표로 작성한다.

| 개념 | 설명 | 중요 이유 |
"""

프롬프트를 생성합니다.

In [ ]:
final_prompt = make_prompt(
    role,
    context,
    task,
    constraints,
    example,
    output_format,
    document
)

print(final_prompt)

모델에 전달합니다.

In [ ]:
result = ask_llm(final_prompt)

print("=== 최종 결과 ===")
print(result)

### 실험 A와 F의 결과를 나란히 비교

In [ ]:
print("=" * 60)
print("A. 모호한 프롬프트")
print("=" * 60)

print(result_a)


print("\n\n")

print("=" * 60)
print("F. 6요소를 사용한 프롬프트")
print("=" * 60)

print(result_f)

## 실습 3. 추론 전략 비교

### 추론 문제 준비

In [ ]:
problem = """
철수는 사과 23개를 가지고 있었다.
그중 8개를 친구에게 주었다.
그 후 사과 6개를 새로 샀다.
남은 사과의 3분의 1을 동생에게 주었다.

철수에게 남은 사과는 몇 개인가?
"""

23 - 8 = 15

15 + 6 = 21

21 ÷ 3 = 7

21 - 7 = 14

정답 = 14

### 모델 실행 함수
실행 시간도 함께 측정합니다.

In [ ]:
import time

def ask_llm(prompt, do_sample=False, temperature=None):
    """
    모델에게 질문하고
    답변과 실행 시간을 반환합니다.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 모델의 채팅 형식으로 변환
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 문자열 → Token ID
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 생성 옵션
    generation_args = {
        "max_new_tokens": 300,
        "do_sample": do_sample
    }

    # sampling을 사용할 때만 temperature 적용
    if do_sample:
        generation_args["temperature"] = temperature

    start = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_args
        )

    elapsed = time.time() - start

    # 새롭게 생성된 토큰만 추출
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer.strip(), elapsed, len(generated)

### 실험 A. Direct Answer
추론 방법을 별도로 지정하지 않습니다.

In [ ]:
prompt_direct = f"""
다음 문제를 풀어라.

{problem}

마지막 줄에 반드시
정답: 숫자
형식으로 출력하라.
"""

answer_direct, time_direct, tokens_direct = ask_llm(
    prompt_direct
)

print("=== A. Direct Answer ===")
print(answer_direct)

print("\n실행 시간:", round(time_direct, 2), "초")
print("생성 토큰:", tokens_direct)

### 실험 B. Step-by-step 추론
이번에는 문제를 중간 단계로 나누어 계산하도록 요청합니다.

In [ ]:
prompt_step = f"""
다음 문제를 풀어라.

{problem}

계산에 필요한 중간 단계를 순서대로 확인한 뒤
최종 답을 구하라.

마지막 줄에 반드시
정답: 숫자
형식으로 출력하라.
"""

answer_step, time_step, tokens_step = ask_llm(
    prompt_step
)

print("=== B. Step-by-step ===")
print(answer_step)

print("\n실행 시간:", round(time_step, 2), "초")
print("생성 토큰:", tokens_step)

### 실험 C. Self-Consistency
이번에는 같은 문제를 5번 풀게 합니다.

In [ ]:
do_sample=True   # 서로 다른 추론 경로가 생성될 수 있게

prompt_sc = f"""
다음 문제를 풀어라.

{problem}

계산에 필요한 중간 단계를 순서대로 확인한 뒤
최종 답을 구하라.

마지막 줄에 반드시
정답: 숫자
형식으로 출력하라.
"""

answers_sc = []

total_time = 0
total_tokens = 0

# 같은 문제를 5번 풉니다.
for i in range(5):

    answer, elapsed, tokens = ask_llm(
        prompt_sc,

        # 다양한 답을 생성하도록 sampling 사용
        do_sample=True,
        temperature=0.8
    )

    answers_sc.append(answer)

    total_time += elapsed
    total_tokens += tokens

    print(f"\n=== 추론 {i+1} ===")
    print(answer)

### 각 답에서 최종 숫자 추출
모델의 긴 답변에서 정답만 추출

In [ ]:
import re

def extract_answer(text):
    """
    '정답: 숫자'에서 숫자만 추출합니다.
    """

    match = re.search(
        r"정답\s*:\s*(-?\d+)",
        text
    )

    if match:
        return int(match.group(1))

    return None

In [ ]:
final_answers = []

for answer in answers_sc:

    number = extract_answer(answer)

    final_answers.append(number)


print("5번의 최종 답:")
print(final_answers)

### 다수결로 최종 답 선택
가장 많이 등장한 답을 선택합니다.

In [ ]:
from collections import Counter

# None은 답 추출에 실패한 경우이므로 제외
valid_answers = [
    x for x in final_answers
    if x is not None
]

counter = Counter(valid_answers)

# 가장 많이 나온 답
majority_answer = counter.most_common(1)[0][0]

print("각 답의 빈도:", counter)

print("Self-Consistency 최종 답:",
      majority_answer)

### 세 방법의 비용 비교

In [ ]:
print("=" * 55)
print("추론 전략 비용 비교")
print("=" * 55)

print(
    f"Direct           : "
    f"{time_direct:.2f}초 / {tokens_direct} tokens"
)

print(
    f"Step-by-step     : "
    f"{time_step:.2f}초 / {tokens_step} tokens"
)

print(
    f"Self-Consistency : "
    f"{total_time:.2f}초 / {total_tokens} tokens"
)